In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity
import json, warnings
warnings.filterwarnings('ignore')

print('Libraries imported!')

In [ ]:
# Load the cleaned ATS pairs dataset
df = pd.read_csv('cleaned_resumeJD_pairs.csv')
print(f'Loaded: {len(df)} pairs')
print(f'\nLabel distribution:')
print(df['match_label'].value_counts())
print(f'\nScore range: {df["match_score"].min():.2f} - {df["match_score"].max():.2f}')
df.head(3)

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.3, random_state=42, stratify=df['match_label']
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['match_label']
)

print(f'Train:      {len(train_df)} pairs')
print(f'Validation: {len(val_df)} pairs')
print(f'Test:       {len(test_df)} pairs')
print()

# Verify each split has all 3 labels
for name, split in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    print(f'{name} label dist: {split["match_label"].value_counts().to_dict()}')

In [ ]:
# Convert to InputExample format — sentence-transformers expects this
# text1 = resume, text2 = job description, label = match_score (float 0-1)

train_examples = [
    InputExample(
        texts=[row['resume_text'], row['job_description']],
        label=float(row['match_score'])
    )
    for _, row in train_df.iterrows()
]

val_examples = [
    InputExample(
        texts=[row['resume_text'], row['job_description']],
        label=float(row['match_score'])
    )
    for _, row in val_df.iterrows()
]

print(f'Train examples: {len(train_examples)}')
print(f'Val examples:    {len(val_examples)}')

print('\nSample InputExample:')
print(f'  text1 (resume): {train_examples[0].texts[0][:200]}...')
print(f'  text2 (JD):     {train_examples[0].texts[1][:200]}...')
print(f'  label:          {train_examples[0].label}')

In [ ]:
print('loading base bert model')
base_model = SentenceTransformer('all-mpnet-bas')
print('model loaded successfully')

In [ ]:
# Evaluate base model on test set BEFORE fine-tuning
# This is our baseline, fine-tuning should beat this
print('Evaluating base model on test set...')

base_preds = []

for _, row in test_df.iterrows():
    emb1 = base_model.encode(row['resume_text'])
    emb2 = base_model.encode(row['job_description'])
    sim = cosine_similarity([emb1], [emb2])[0][0]
    base_preds.append(float(sim))

base_mae = mean_absolute_error(test_df['match_score'], base_preds)
base_rmse = np.sqrt(mean_squared_error(test_df['match_score'], base_preds))

print(f'\nBase Model - Test Set Performance')
print(f'  MAE:  {base_mae:.4f}')
print(f'  RMSE: {base_rmse:.4f}')
print()
print('Goal: fine-tuning should reduce MAE below this baseline.')

In [ ]:
model = SentenceTransformer('all-mpnet-base-v2')

train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=16
)

train_loss = losses.CosineSimilarityLoss(model)

evaluator = EmbeddingSimilarityEvaluator.from_input_examples(
    val_examples,
    name='ats-val'
)

print('Training setup ready.')
print(f'  Batch size: 16')
print(f'  Train pairs: {len(train_examples)}')
print(f'  Steps/epoch: {len(train_dataloader)}')

In [ ]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=10,
    evaluation_steps=len(train_dataloader),
    warmup_steps=warmup_steps,
    output_path='models/finetuned-bert',
    save_best_model=True,
    save_steps=total_steps,
    use_amp=True,
    show_progress_bar=True
)

print('\n FINE TUNING COMPLETE!!')

In [ ]:
finetuned_model = SentenceTransformer('models/finetuned-bert')

print('Fine-tuned model loaded.')

In [ ]:
ft_preds = []

for _, row in test_df.iterrows():
    emb1 = finetuned_model.encode(row['resume_text'])
    emb2 = finetuned_model.encode(row['job_description'])
    sim = cosine_similarity([emb1], [emb2])[0][0]
    ft_preds.append(float(sim))

ft_mae = mean_absolute_error(test_df['match_score'], ft_preds)
ft_rmse = np.sqrt(mean_squared_error(test_df['match_score'], ft_preds))

print(f'\nFine-tuned Model - Test Set Performance')
print(f'  MAE:  {ft_mae:.4f}')
print(f'  RMSE: {ft_rmse:.4f}')

In [ ]:
# Side-by-side comparison
print('=' * 50)
print('MODEL COMPARISON')
print('=' * 50)
print(f'  Base MAE:        {base_mae:.4f}')
print(f'  Fine-tuned MAE:  {ft_mae:.4f}')
print(f'  Improvement:     {(base_mae - ft_mae) / base_mae * 100:.1f}%')
print('=' * 50)

In [ ]:
# Scatter plots: base vs fine-tuned
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = test_df['match_label'].map({'low': 'red', 'medium': 'orange', 'high': 'green'})

for ax, preds, title, mae in [
    (axes[0], base_preds, 'Base Model', base_mae),
    (axes[1], ft_preds, 'Fine-tuned Model', ft_mae)
]:
    ax.scatter(test_df['match_score'], preds, c=colors, alpha=0.7)
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect')
    ax.set_xlabel('Ground Truth match_score')
    ax.set_ylabel('Predicted Similarity')
    ax.set_title(f'{title} (MAE: {mae:.4f})')
    ax.grid(True, alpha=0.3)

from matplotlib.patches import Patch

fig.legend(handles=[
    Patch(color='green', label='high'),
    Patch(color='orange', label='medium'),
    Patch(color='red', label='low')
], loc='lower center', ncol=3)

plt.tight_layout()
plt.show()

In [ ]:
metadata = {
    'base_model': 'all-mpnet-base-v2',
    'dataset': 'merged_dataset_clean.csv',
    'total_pairs': len(df),
    'train_pairs': len(train_df),
    'val_pairs': len(val_df),
    'test_pairs': len(test_df),
    'epochs': 10,
    'batch_size': 16,
    'base_mae': round(float(base_mae), 4),
    'finetuned_mae': round(float(ft_mae), 4),
    'improvement_pct': round((base_mae - ft_mae) / base_mae * 100, 2)
}

with open('models/finetuned-bert/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved: models/finetuned-bert/metadata.json')
print(json.dumps(metadata, indent=2))

In [ ]:
# Production Pipeline Test

def score_resume_against_jd(resume_text, jd_text, model):
    emb_resume = model.encode(resume_text, convert_to_numpy=True)
    emb_jd = model.encode(jd_text, convert_to_numpy=True)

    score = cosine_similarity([emb_resume], [emb_jd])[0][0]

    return float(score)


# Test with realistic cases
test_cases = [
    {
        'label': 'HIGH match expected',
        'resume': 'Senior Python developer with experience in FastAPI, Django, PostgreSQL, AWS, Docker, and REST APIs.',
        'jd': 'We need a Python backend developer with experience in FastAPI, Django, PostgreSQL, AWS, Docker, and REST APIs.'
    },
    {
        'label': 'MEDIUM match expected',
        'resume': 'Python developer with experience in data analysis, Pandas, NumPy, and machine learning.',
        'jd': 'Looking for a backend developer with Python, FastAPI, PostgreSQL, Docker, and cloud deployment experience.'
    },
    {
        'label': 'LOW match expected',
        'resume': 'Frontend developer experienced in React, JavaScript, HTML, CSS, and responsive web design.',
        'jd': 'Looking for a machine learning engineer with Python, PyTorch, TensorFlow, NLP, and deep learning experience.'
    }
]

for case in test_cases:
    score = score_resume_against_jd(
        case['resume'],
        case['jd'],
        finetuned_model
    )

    print(f"{case['label']}: {score:.4f}")